# 05 — Undervalued teams vs Polymarket TI-2026 odds

Combines: model P(champion) (nb 03) + Shin-de-vigged bookmaker consensus +
live Polymarket prices (gamma API) → deviation table with the cost hurdle.

Bias checklist applied (see `research/STRATEGY_SELECTION.md`):
- **FLB**: at 1mo+ horizon longshots are systematically *overpriced* —
  cheap-looking 3–8¢ teams are usually correctly cheap.
- **Settlement wedge**: ~2 months lockup ≈ 0.6–0.8% — subtract from any edge.
- **Fees**: check the market's `feeType`; sports 3% ⇒ peak ~0.75% taker, 0 maker.
- **TI base rate**: pre-tournament favourites have historically won TI rarely —
  wide CIs are honest, not a bug.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 160)
DATA = ROOT / 'data'


In [ ]:
import requests
# Find the TI-2026 winner market on Polymarket (edit slug if needed)
CANDIDATE_SLUGS = ['dota-2-the-international-champions', 'dota-2-the-international-2026']
ev = None
for slug in CANDIDATE_SLUGS:
    r = requests.get('https://gamma-api.polymarket.com/events', params={'slug': slug}, timeout=30).json()
    if r:
        ev = r[0]; break
assert ev, 'TI winner event not found - update CANDIDATE_SLUGS'
rows = []
import json as _json
for mk in ev['markets']:
    if mk.get('closed'): continue
    prices = _json.loads(mk.get('outcomePrices','[]') or '[]')
    rows.append({'team': mk['groupItemTitle'] if 'groupItemTitle' in mk else mk['question'],
                 'pm_price': float(prices[0]) if prices else None,
                 'volume': float(mk.get('volume') or 0),
                 'spread': mk.get('spread')})
pm = pd.DataFrame(rows).sort_values('pm_price', ascending=False)
print(ev['title'], '| feeType:', ev.get('feeType') or (ev['markets'][0].get('feeType')), '| sum of mids:', pm.pm_price.sum().round(4))
pm

In [ ]:
# Bookmaker consensus (EDIT: paste current decimal odds, e.g. from Pinnacle/GG.bet)
BOOK_ODDS = {
    # 'Team Spirit': 3.5, 'Team Liquid': 6.0, 'Xtreme Gaming': 7.0, ...
}
from src.model import shin_devig
books = shin_devig(BOOK_ODDS) if BOOK_ODDS else None
books

In [ ]:
model = pd.read_parquet(DATA / 'model_pchamp.parquet')
t = pm.merge(model.rename(columns={'p_champion':'model_p'}), on='team', how='left')
if books is not None:
    t = t.merge(books[['team','shin_devig']], on='team', how='left')
HOLD_MONTHS = 2.0        # to TI final
WEDGE = 0.0376 * HOLD_MONTHS/12   # lockup opportunity cost
t['fee'] = 0.03 * t.pm_price * (1 - t.pm_price)   # verify feeType first!
t['fair'] = t[['model_p','shin_devig']].mean(axis=1, skipna=True) if books is not None else t.model_p
t['edge'] = t.fair - t.pm_price
t['edge_net_taker'] = t.edge - t.fee - t.pm_price*WEDGE - (t.spread.fillna(0.01)/2)
t['edge_net_maker'] = t.edge - t.pm_price*WEDGE
und = t.sort_values('edge_net_maker', ascending=False)
und[['team','pm_price','model_p','shin_devig','fair','edge','edge_net_taker','edge_net_maker']].round(4) if books is not None else \
und[['team','pm_price','model_p','fair','edge','edge_net_taker','edge_net_maker']].round(4)

**Decision rule** (per `STRATEGY_SELECTION.md`): a team is genuinely
undervalued only if `edge_net_maker` is positive **and** the model and the
de-vigged books *agree* on the direction (consistency test). One-source
edges — especially on sub-10¢ longshots — are presumed to be FLB noise.
Enter maker-only, between matches, cluster-capped.